# Laboratorio 01 â€” Pipeline Declarativa BÃ¡sica con Lakeflow

**Semana:** 05 | **Actividad de referencia:** Actividad 01  
**Modalidad:** Individual | **Entorno:** Databricks Lakeflow (Spark Declarative Pipelines)

---

## Instrucciones generales

Crea una pipeline declarativa bÃ¡sica con `pyspark.pipelines` (`dp`) para tu dataset propio. La pipeline debe incluir al menos una `materialized_view` (batch) y una `table` (streaming o batch), ademÃ¡s de demostrar el uso de parÃ¡metros de pipeline.

> **Importante:** Este notebook debe ejecutarse desde un Databricks Pipeline (Lakeflow), no directamente como notebook interactivo. Las celdas de cÃ³digo son la definiciÃ³n de la pipeline.

## Parte 1 â€” DescripciÃ³n del dataset y diseÃ±o de la pipeline

1. **Nombre, fuente y URL** del dataset.
2. **Â¿QuÃ© representarÃ¡ la materialized_view?** Â¿Un resumen, un join, una vista limpia de los datos?
3. **Â¿QuÃ© representarÃ¡ la table?** Â¿La carga incremental, los datos filtrados, un agrupamiento?
4. **ParÃ¡metros de pipeline:** Â¿QuÃ© valores querrÃ­as parametrizar? (ruta de origen, nombres de tabla, umbrales, etc.)
5. **Preguntas de negocio** que esta pipeline ayudarÃ­a a responder.

**Escribe tu respuesta aquÃ­:**

## Parte 2 â€” Importaciones y parÃ¡metros de pipeline

In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# ParÃ¡metros de pipeline: se configuran en la UI de Lakeflow (Settings â†’ Parameters)
# AquÃ­ solo se leen; los valores se pasan en el job de la pipeline
RUTA_ORIGEN  = spark.conf.get("ruta_origen",  "/Volumes/workspace/default/week_5/tu_archivo.csv")
ENTORNO      = spark.conf.get("entorno",       "dev")
UMBRAL_MIN   = float(spark.conf.get("umbral_minimo", "0"))

print(f"ParÃ¡metros de pipeline:")
print(f"  ruta_origen   = {RUTA_ORIGEN}")
print(f"  entorno       = {ENTORNO}")
print(f"  umbral_minimo = {UMBRAL_MIN}")

## Parte 3 â€” Perfil tÃ©cnico del dataset

> Ejecuta esta secciÃ³n como notebook interactivo (fuera de la pipeline) para entender el dataset antes de definir las transformaciones declarativas.

In [ ]:
# ExploraciÃ³n interactiva â€” NO usar decoradores @dp aquÃ­
df_explorar = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(RUTA_ORIGEN)

print(f"Filas: {df_explorar.count():,} | Columnas: {len(df_explorar.columns)}")
df_explorar.printSchema()
df_explorar.show(5, truncate=False)

In [ ]:
# Nulos por columna
total = df_explorar.count()
df_explorar.select([
    F.round(
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) * 100.0 / total, 1
    ).alias(f"{c}_pct_nulos")
    for c in df_explorar.columns
]).show(truncate=False)

In [ ]:
# Cardinalidades de columnas categÃ³ricas
# Reemplaza 'columna_categorica' con el nombre real de tu columna
df_explorar.groupBy("columna_categorica").count().orderBy(F.col("count").desc()).show(15)

**Observaciones del perfil:**  
Â¿QuÃ© columnas tienen nulos que deberÃ­as manejar en la pipeline? Â¿QuÃ© columna categÃ³rica tiene mejor distribuciÃ³n para usarla como particiÃ³n?

## Parte 4 â€” DefiniciÃ³n declarativa de la pipeline

Estas celdas constituyen la pipeline en sÃ­. Cuando se ejecuten desde Lakeflow, `@dp.materialized_view()` y `@dp.table()` registran las transformaciones en el grafo de la pipeline.

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Bronze: lectura raw â€” materialized_view
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@dp.materialized_view(
    comment="Carga raw del dataset desde el volumen. Sin transformaciones."
)
def bronze_mi_dataset():
    return (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(RUTA_ORIGEN)
        .withColumn("_ingest_ts", F.current_timestamp())
        .withColumn("_entorno",   F.lit(ENTORNO))
    )

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Silver: limpieza y filtrado â€” materialized_view
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@dp.materialized_view(
    comment="Silver: registros filtrados, tipos correctos, strings normalizados."
)
def silver_mi_dataset():
    return (
        dp.read("bronze_mi_dataset")
        .filter(F.col("columna_clave").isNotNull())
        .filter(F.col("columna_numerica") >= UMBRAL_MIN)
        .withColumn("columna_texto", F.trim(F.lower(F.col("columna_texto"))))
        # AÃ±ade aquÃ­ tus propias transformaciones de limpieza
        .withColumn("_silver_ts", F.current_timestamp())
    )

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Gold: agregaciÃ³n de negocio â€” table (persistida)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@dp.table(
    comment="Gold: KPIs por categorÃ­a calculados desde Silver."
)
def gold_kpis_mi_dataset():
    return (
        dp.read("silver_mi_dataset")
        .groupBy("columna_categorica")
        .agg(
            F.count("*").alias("total_registros"),
            F.avg("columna_numerica").alias("promedio"),
            F.max("columna_numerica").alias("maximo"),
            F.min("columna_numerica").alias("minimo")
        )
        .orderBy(F.col("total_registros").desc())
    )

**AnÃ¡lisis de la pipeline:**
1. Â¿CuÃ¡ntos nodos tiene el grafo de tu pipeline (Bronze â†’ Silver â†’ Gold)?
2. Â¿CuÃ¡l es la diferencia entre `@dp.materialized_view()` y `@dp.table()`? Â¿CuÃ¡ndo usarÃ­as cada uno?
3. Â¿QuÃ© pasa si falla `silver_mi_dataset`? Â¿La pipeline intenta ejecutar `gold_kpis_mi_dataset` de todas formas?

## Parte 5 â€” ConfiguraciÃ³n de la pipeline (teÃ³rico)

Describe en markdown cÃ³mo configurarÃ­as esta pipeline en la UI de Lakeflow.

```json
{
  "name": "lab05-01-pipeline-mi-dataset",
  "target": "workspace.default",
  "clusters": [{"num_workers": 1}],
  "libraries": [
    {"notebook": {"path": "/semana_05/laboratorios/lab_01_pipeline_basica"}}
  ],
  "configuration": {
    "ruta_origen":    "/Volumes/workspace/default/week_5/tu_archivo.csv",
    "entorno":        "dev",
    "umbral_minimo":  "0"
  },
  "mode": "TRIGGERED"
}
```

**Preguntas:**
1. Â¿QuÃ© diferencia hay entre modo `TRIGGERED` y `CONTINUOUS`?
2. Â¿CÃ³mo cambia la config si quisieras procesar datos en tiempo real con Auto Loader?

## Parte 6 â€” Preguntas de negocio sobre los resultados Gold

DespuÃ©s de ejecutar la pipeline, consulta los resultados desde un notebook interactivo separado.

In [ ]:
# Ejecutar esta celda en un notebook interactivo DESPUÃ‰S de correr la pipeline
# spark.table("workspace.default.gold_kpis_mi_dataset").show(15, truncate=False)

**Preguntas de negocio** (responde despuÃ©s de ejecutar la pipeline):
1. Â¿QuÃ© categorÃ­a tiene el mayor promedio? Â¿Lo esperabas?
2. Â¿CuÃ¡ntos registros se perdieron de Bronze a Silver? Â¿Por quÃ©?
3. Â¿QuÃ© hallazgo en Gold te darÃ­a mÃ¡s impacto si lo presentaras a un stakeholder?

## Parte 7 â€” ReflexiÃ³n final

1. Â¿QuÃ© ventaja tiene definir la pipeline con decoradores vs con un script imperativo que llama a `.write()`?
2. Â¿CÃ³mo maneja Lakeflow las dependencias entre `materialized_view` y `table` automÃ¡ticamente?
3. Â¿QuÃ© pasa si modificas la lÃ³gica de `silver_mi_dataset` y re-ejecutas la pipeline? Â¿Recomputa Gold tambiÃ©n?
4. Â¿CÃ³mo agregarÃ­as una expectativa (`@dp.expect`) para detectar registros de baja calidad sin detener la pipeline?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_05/laboratorios/lab_01_pipeline_basica.ipynb semana_05/laboratorios/<tu-nombre>/lab_01_pipeline_basica.ipynb

git add semana_05/laboratorios/<tu-nombre>/lab_01_pipeline_basica.ipynb
git commit -m "lab: semana05 lab01 pipeline basica materialized_view table <nombre-dataset> - <tu-nombre>"
git push origin develop
```